In [1]:
import pandas as pd, numpy as np, xgboost as xgb, json, mlflow
from sklearn.metrics import average_precision_score, roc_auc_score

feast_feats = ["hour", "day_of_week", "tx_count_before", "secs_since_last",
               "amt_vs_avg", "device_changed", "addr_changed", "email_changed"]
msg_feats = (["TransactionAmt", "card1", "card2", "card3", "card5", "addr1", "addr2", "dist1"]
             + [f"C{i}" for i in range(1, 15)] + [f"D{i}" for i in range(1, 16)])
all_feats = feast_feats + msg_feats

f = pd.read_parquet("data/features.parquet")
raw = pd.read_csv("data/train_transaction.csv", usecols=["TransactionID"] + msg_feats)
df = f.merge(raw, on="TransactionID").sort_values("event_timestamp").reset_index(drop=True)

n = len(df); a, b = int(n * 0.72), int(n * 0.80)
X, y = df[all_feats], df["isFraud"]
ratio = (y[:a] == 0).sum() / (y[:a] == 1).sum()

m = xgb.XGBClassifier(n_estimators=1000, max_depth=6, learning_rate=0.05, subsample=0.8,
                      colsample_bytree=0.8, scale_pos_weight=ratio, tree_method="hist",
                      eval_metric="aucpr", early_stopping_rounds=50, random_state=42)
m.fit(X[:a], y[:a], eval_set=[(X[a:b], y[a:b])], verbose=100)

p = m.predict_proba(X[b:])[:, 1]
pr, roc = average_precision_score(y[b:], p), roc_auc_score(y[b:], p)
print("serving model  PR-AUC:", round(pr, 4), " ROC-AUC:", round(roc, 4), " rounds:", m.best_iteration)

m.save_model("models/xgb_serving.json")
json.dump({"feast": feast_feats, "message": msg_feats}, open("models/serving_features.json", "w"))

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("fraud-detection")
with mlflow.start_run(run_name="xgb_serving"):
    mlflow.log_metrics({"pr_auc": pr, "roc_auc": roc})

[0]	validation_0-aucpr:0.35225
[100]	validation_0-aucpr:0.49017
[200]	validation_0-aucpr:0.50812
[300]	validation_0-aucpr:0.52337
[400]	validation_0-aucpr:0.53360
[500]	validation_0-aucpr:0.54283
[600]	validation_0-aucpr:0.54916
[700]	validation_0-aucpr:0.55179
[800]	validation_0-aucpr:0.55506
[900]	validation_0-aucpr:0.55732
[999]	validation_0-aucpr:0.56073
serving model  PR-AUC: 0.4584  ROC-AUC: 0.8808  rounds: 993


In [2]:
%%writefile src/producer.py
import json, time, sys
import pandas as pd
from kafka import KafkaProducer

RATE = float(sys.argv[1]) if len(sys.argv) > 1 else 5
LIMIT = int(sys.argv[2]) if len(sys.argv) > 2 else 200

msg_feats = (["TransactionAmt", "card1", "card2", "card3", "card5", "addr1", "addr2", "dist1"]
             + [f"C{i}" for i in range(1, 15)] + [f"D{i}" for i in range(1, 16)])

f = pd.read_parquet("data/features.parquet", columns=["TransactionID", "card_id", "event_timestamp", "isFraud"])
raw = pd.read_csv("data/train_transaction.csv", usecols=["TransactionID"] + msg_feats)
df = f.merge(raw, on="TransactionID").sort_values("event_timestamp")
df = df.iloc[int(len(df) * 0.8):].head(LIMIT)

producer = KafkaProducer(bootstrap_servers="localhost:9092",
                         value_serializer=lambda v: json.dumps(v).encode("utf-8"))

for _, r in df.iterrows():
    msg = {"transaction_id": int(r.TransactionID), "card_id": int(r.card_id),
           "true_label": int(r.isFraud)}
    for c in msg_feats:
        msg[c] = None if pd.isna(r[c]) else float(r[c])
    producer.send("transactions", msg)
    time.sleep(1 / RATE)

producer.flush()
print("sent", len(df), "messages")

Overwriting src/producer.py
